## Setup — run this first

Mounts Drive, points the notebook at your project folder, and installs
what's missing. No git, no tokens.

**Your Drive folder must look like this:**

```
MyDrive/Ghana_Dropout_Project_R02/
├── config.py          <- these three at the TOP level,
├── losses.py             not inside notebooks/
├── pipeline.py
├── requirements.txt
├── notebooks/         <- the 11 notebooks
└── data-raw/
    └── ghana_dropout_study_M.xlsx
```

`results/`, `figures/`, `models/` and `data-processed/` are created for you.

Drive saves as it goes, so there is nothing to push — but see the checklist
in the last cell before you submit.


In [11]:
# ============================================================
# SETUP — Google Drive. Run first. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

PROJECT = "/content/drive/MyDrive/Ghana_Dropout_Project_R02"   # <-- edit if yours differs
RAW_XLSX_NAME = "ghana_dropout_study_M.xlsx"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    root = Path(PROJECT)
    if not root.exists():
        raise FileNotFoundError(
            f"{PROJECT} does not exist.\n"
            "Create that folder in My Drive and put config.py, losses.py, "
            "pipeline.py, requirements.txt, the notebooks/ folder and "
            "data-raw/ inside it."
        )

    # the three modules must sit at the project root, not in notebooks/
    missing = [m for m in ("config.py", "losses.py", "pipeline.py")
               if not (root / m).exists()]
    if missing:
        stray = [m for m in missing if (root / "notebooks" / m).exists()]
        msg = f"Missing from {PROJECT}: {missing}"
        if stray:
            msg += (f"\n{stray} are in notebooks/ instead. Move them UP one "
                    "level, into the project folder itself. If they stay in "
                    "notebooks/, that folder gets treated as the project root "
                    "and results/ is written in the wrong place.")
        raise FileNotFoundError(msg)

    os.chdir(root)
    os.environ["DROPOUT_REPO"] = str(root)

    # Forget any previously loaded copy of the project modules. Python keeps
    # the first version it imported for the whole session, so an edited
    # config.py is silently ignored until the runtime restarts. This makes
    # every run use the files currently in Drive.
    for _m in ("config", "losses", "pipeline"):
        sys.modules.pop(_m, None)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    # ---- dependencies: only install what is actually missing ------------
    need = []
    for mod, pkg in [("lightgbm", "lightgbm"), ("shap", "shap"),
                     ("catboost", "catboost"), ("xgboost", "xgboost"),
                     ("imblearn", "imbalanced-learn"), ("openpyxl", "openpyxl")]:
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print("installing:", need)
        subprocess.run(f"pip install -q {' '.join(need)}", shell=True)
    else:
        print("all dependencies present")

    # ---- raw workbook ---------------------------------------------------
    (root / "data-raw").mkdir(exist_ok=True)
    xlsx = root / "data-raw" / RAW_XLSX_NAME
    if xlsx.exists():
        print(f"raw workbook: {xlsx.name}")
    else:
        loose = list(root.glob(RAW_XLSX_NAME)) + list(root.glob(f"**/{RAW_XLSX_NAME}"))
        if loose:
            import shutil
            shutil.copy(loose[0], xlsx)
            print(f"copied {loose[0]} -> data-raw/")
        else:
            print(f"NOT FOUND: data-raw/{RAW_XLSX_NAME}\n"
                  "Notebook 1 needs it. Notebooks 2-9 read "
                  "data-processed/cleaned_data.csv instead and are fine "
                  "without it.")

    print(f"\nPROJECT : {os.getcwd()}")
else:
    print("Not in Colab — paths resolve from the project root.")


all dependencies present
raw workbook: ghana_dropout_study_M.xlsx

PROJECT : /content/drive/MyDrive/Ghana_Dropout_Project_R02


# Notebook 2 — Exploratory Data Analysis

## What changed from R01

**EDA now runs on the training pool only.** The frozen 80/20 split happens
first, and the 200 test rows are not described, plotted or correlated here.

This is stricter than the examination required, and worth a sentence in M6.
Analyst-level peeking is a real leakage channel: if you choose composite
weights, or notice an outlier, or decide a feature matters after looking at a
plot that included the test rows, that decision carries test information into
the model even though no `.fit()` touched them. Since GATE-1 already failed on
mechanical leakage, the cheapest way to make the corrected pipeline credible
is for nothing at all to have looked at the test partition before Notebook 8.

Also new here: the plots that answer questions the examination asked and R01
had no figure for — school-level dropout rates, the income-category audit,
the attendance out-of-range distribution, and where the base rate sits.

In [12]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
from config import *
from pipeline import frozen_split, binarise_target

banner("NOTEBOOK 2 — EDA (TRAINING POOL ONLY)")
OUT = run_dir("notebook02_eda")
FIGS = OUT / "figures"
print("outputs ->", OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

print(f"full           : {len(df)} rows")
print(f"train_pool     : {len(train_pool)} rows, "
      f"{int(train_pool[TARGET].sum())} dropout "
      f"({100*train_pool[TARGET].mean():.1f}%)")
print(f"test_holdout   : {len(test_holdout)} rows "
      "-- NOT described below, not plotted, not touched until Notebook 8")

eda = train_pool          # everything from here uses eda, never df
BASE_RATE = float(eda[TARGET].mean())

NOTEBOOK 2 — EDA (TRAINING POOL ONLY)
repo            : /content/drive/MyDrive/Ghana_Dropout_Project_R02
provenance      : NONE — set FREEZE_TAG in config.py before scoring the test set
school_handling : drop
FEATURE SET     : records   (PRIMARY — school records only)
primary metric  : auc_pr
outputs -> /content/drive/MyDrive/Ghana_Dropout_Project_R02/results/notebook02_eda/20260921T134434Z_records
full           : 981 rows
train_pool     : 784 rows, 69 dropout (8.8%)
test_holdout   : 197 rows -- NOT described below, not plotted, not touched until Notebook 8


In [14]:
# ---- 1. shape, dtypes, target -------------------------------------------
print(eda.dtypes.value_counts().to_string())
num = [c for c in eda.columns if not is_text(eda[c])]
cat = [c for c in eda.columns if c not in num]
print(f"\nnumeric: {len(num)}   categorical: {len(cat)}")

desc = eda[num].describe().T
desc.to_csv(OUT / "descriptive_statistics.csv")
print("\nTable 2 source (report min/max honestly — out-of-range values are "
      "visible here and treated in-fold):")
print(desc[["mean", "std", "min", "max"]].round(3).to_string())

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=eda, x=TARGET, ax=ax)
ax.set_title(f"Target distribution (training pool, base rate {100*BASE_RATE:.1f}%)")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout(); plt.savefig(FIGS / "target_distribution.png", dpi=200); plt.close()
print(f"\nabsolute positive count in the training pool: {int(eda[TARGET].sum())}")
print("Carry this number beside every reported rate (J4).")

object     27
float64    14
int64       1

numeric: 15   categorical: 27

Table 2 source (report min/max honestly — out-of-range values are visible here and treated in-fold):
                                 mean     std   min      max
age_at_start_of_academic_year  11.377   2.317   5.0   15.300
term_1_attendance              81.235  15.728  20.0  100.500
term_2_attendance              83.438  13.458  10.0  102.900
term_3_attendance              81.146  19.954   0.0   99.000
average_attendance             81.963  15.967  10.0   98.667
english_exam_score             67.562  14.507  11.0   95.000
math_exam_score                65.151  15.750   2.0   95.000
science_exam_score             67.700  13.729  11.0   96.000
average_exam_score             67.148  11.169  20.5   94.000
safety_at_home                  3.622   0.999   0.7    5.700
safety_at_school                3.787   1.134   0.4    5.700
teacher_support_rating          3.690   1.031   1.0    6.000
school_enjoyment                

In [15]:
# ---- 2. school-level structure (GATE-1 iv, Q2) --------------------------
if SCHOOL_COL in eda.columns:
    g = (eda.groupby(SCHOOL_COL)
           .agg(n=(TARGET, "size"), n_dropout=(TARGET, "sum")))
    g["rate_pct"] = (100 * g["n_dropout"] / g["n"]).round(1)
    g = g.sort_values("n", ascending=False)
    g.to_csv(OUT / "school_structure.csv")
    print(g.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    g["n"].plot.bar(ax=axes[0], color="steelblue")
    axes[0].axhline(len(eda)/len(g), ls="--", c="k", lw=1, label="even split")
    axes[0].set_title("Records per school"); axes[0].legend()
    g["rate_pct"].plot.bar(ax=axes[1], color="indianred")
    axes[1].axhline(100*BASE_RATE, ls="--", c="k", lw=1, label="overall base rate")
    axes[1].set_title("Dropout rate per school (%)"); axes[1].legend()
    plt.tight_layout(); plt.savefig(FIGS / "school_structure.png", dpi=200); plt.close()

    print(f"\n{g.shape[0]} schools. Largest holds {100*g['n'].iloc[0]/len(eda):.0f}%. "
          f"Rates span {g['rate_pct'].min():.1f}%-{g['rate_pct'].max():.1f}%.")
    print("This is the figure to put in your own limitations paragraph before "
          "a reviewer finds it in the data file.")

               n  n_dropout  rate_pct
school_code                          
WEWE         381         39      10.2
KNU_JHS      263          9       3.4
SHI           73         15      20.5
AYED_RC       67          6       9.0

4 schools. Largest holds 49%. Rates span 3.4%-20.5%.
This is the figure to put in your own limitations paragraph before a reviewer finds it in the data file.


In [16]:
# ---- 3. income category audit (Q3) -------------------------------------
inc = SOCIOECONOMIC_COLS["family_income"]
if inc in eda.columns:
    vc = eda[inc].value_counts(dropna=False)
    print(f"{inc} as recorded:\n{vc.to_string()}")
    print(f"\nafter canonicalisation -> {sorted(set(CATEGORY_CANONICAL[inc].values()))}")
    print(f"ordinal map -> {ORDINAL_MAPS.get(inc)}")
    print("\n'Unknown' maps to NaN plus a separate indicator, NOT to Medium. "
          "The R01 Notebook 3 mapped \"don't know\" to 1 (Medium), which "
          "silently imputes a value and hides the non-response.")

    ct = pd.crosstab(eda[inc], eda[TARGET], normalize="index") * 100
    print("\ndropout % by recorded income category:")
    print(ct.round(1).to_string())

family_income_level as recorded:
family_income_level
Medium        449
High          204
Low            98
Don't know     30
Hgh             3

after canonicalisation -> ['High', 'Low', 'Medium']
ordinal map -> {'Low': 0, 'Medium': 1, 'High': 2}

'Unknown' maps to NaN plus a separate indicator, NOT to Medium. The R01 Notebook 3 mapped "don't know" to 1 (Medium), which silently imputes a value and hides the non-response.

dropout % by recorded income category:
dropout_label            0     1
family_income_level             
Don't know           100.0   0.0
Hgh                  100.0   0.0
High                  99.5   0.5
Low                   40.8  59.2
Medium                97.8   2.2


In [17]:
# ---- 4. attendance distributions and the >100% cases -------------------
present = [c for c in ATTENDANCE_COLS if c in eda.columns]
if present:
    fig, axes = plt.subplots(1, len(present), figsize=(4*len(present), 3.5))
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, present):
        s = pd.to_numeric(eda[c], errors="coerce")
        sns.histplot(s, ax=ax, bins=30)
        ax.axvline(ATTENDANCE_MAX, ls="--", c="r", lw=1)
        n_over = int((s > ATTENDANCE_MAX).sum())
        ax.set_title(f"{c}\n{n_over} values > 100%")
    plt.tight_layout(); plt.savefig(FIGS / "attendance_distributions.png", dpi=200)
    plt.close()
    for c in present:
        s = pd.to_numeric(eda[c], errors="coerce")
        print(f"{c:22s} min={s.min():7.1f} max={s.max():7.1f} "
              f">100: {int((s>ATTENDANCE_MAX).sum()):4d}")

term_1_attendance      min=   20.0 max=  100.5 >100:    1
term_2_attendance      min=   10.0 max=  102.9 >100:    2
term_3_attendance      min=    0.0 max=   99.0 >100:    0


In [18]:
# ---- 5. correlations and the near-perfect-separation question ----------
# The sharpest thing in the examination: a model at 0.995 accuracy with one
# error on 200 pupils does not look like any real dropout problem. Before
# modelling, check whether a single raw feature already separates the classes.
corr = eda[num].corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, square=False)
plt.title("Correlation matrix (training pool)")
plt.tight_layout(); plt.savefig(FIGS / "correlation_matrix.png", dpi=200); plt.close()

tc = (corr[TARGET].drop(TARGET).abs().sort_values(ascending=False))
print("absolute correlation with the target, top 15:")
print(tc.head(15).round(3).to_string())
tc.to_csv(OUT / "target_correlations.csv")

print("\nSINGLE-FEATURE SEPARATION CHECK")
print("AUC-PR achievable from each feature alone (training pool):")
from sklearn.metrics import average_precision_score
single = []
y = eda[TARGET].to_numpy()
for c in num:
    if c == TARGET:
        continue
    s = pd.to_numeric(eda[c], errors="coerce").fillna(eda[c].median())
    ap = max(average_precision_score(y, s), average_precision_score(y, -s))
    single.append({"feature": c, "auc_pr_alone": ap})
single = pd.DataFrame(single).sort_values("auc_pr_alone", ascending=False)
single["base_rate"] = BASE_RATE
single.to_csv(OUT / "single_feature_separation.csv", index=False)
print(single.head(12).round(4).to_string(index=False))
print(f"\nbase rate (an uninformative feature scores about this): {BASE_RATE:.4f}")
print("\nIf one feature alone reaches ~0.9 AUC-PR, that feature is carrying the "
      "separation and the modelling contribution is small. THAT is the finding "
      "to chase, and it is a better paper than a null on a loss function. "
      "If it is school_code, you have a cluster artefact, not a pupil-risk model.")

absolute correlation with the target, top 15:
average_attendance               0.759
term_3_attendance                0.754
term_1_attendance                0.752
term_2_attendance                0.702
average_exam_score               0.690
english_exam_score               0.571
science_exam_score               0.567
math_exam_score                  0.544
class_participation              0.402
teacher_support_rating           0.341
safety_at_home                   0.311
safety_at_school                 0.229
school_enjoyment                 0.225
age_at_start_of_academic_year    0.205

SINGLE-FEATURE SEPARATION CHECK
AUC-PR achievable from each feature alone (training pool):
               feature  auc_pr_alone  base_rate
    average_exam_score        0.9375      0.088
    average_attendance        0.9006      0.088
     term_1_attendance        0.8800      0.088
     term_3_attendance        0.8698      0.088
    science_exam_score        0.8440      0.088
     term_2_attendance      

In [19]:
# ---- 6. save -----------------------------------------------------------
write_manifest(OUT, {
    "notebook": "02_eda",
    "scope": "training pool only; test partition untouched",
    "train_pool_n": int(len(train_pool)),
    "train_pool_positive": int(train_pool[TARGET].sum()),
    "base_rate": BASE_RATE,
    "top_single_feature": single.iloc[0].to_dict() if len(single) else None,
})
print("figures ->", FIGS)
print("NEXT: Notebook 3 (feature engineering as a fold-safe pipeline).")

figures -> /content/drive/MyDrive/Ghana_Dropout_Project_R02/results/notebook02_eda/20260921T134434Z_records/figures
NEXT: Notebook 3 (feature engineering as a fold-safe pipeline).


---

## Before you submit

Drive has already saved everything — nothing to push. But two things still
have to happen before submission, and neither is automatic.


In [20]:
# ---- what this run produced, and what is still owed ----
import os, sys
from pathlib import Path

try:
    latest = sorted(Path(OUT).parent.glob("*"))[-1]
    files = sorted(p.relative_to(OUT).as_posix() for p in Path(OUT).rglob("*")
                   if p.is_file())
    print(f"run directory : {Path(OUT).relative_to(REPO)}")
    print(f"files written : {len(files)}")
    for f in files:
        print("   ", f)
except Exception as e:
    print("no run directory recorded in this session:", e)

print("""
────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all of results/

2. SET config.FREEZE_TAG BEFORE SCORING THE TEST SET.
   Without git there is no commit hash to anchor the freeze to. Put a
   fixed dated string in config.py — e.g. "R02-freeze-2026-09-25-1430" —
   at the moment you freeze the configuration, and never revise it.
   Notebook 8 refuses to score the test set until it is set.
────────────────────────────────────────────────────────────────────""")


run directory : results/notebook02_eda/20260921T134434Z_records
files written : 9
    RUN_MANIFEST.json
    descriptive_statistics.csv
    figures/attendance_distributions.png
    figures/correlation_matrix.png
    figures/school_structure.png
    figures/target_distribution.png
    school_structure.csv
    single_feature_separation.csv
    target_correlations.csv

────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.